## Example 1: Maturin CLI and Version Information

Demonstrate basic maturin CLI access and version information.

In [ ]:
# Example 1: Maturin CLI and version information
import subprocess
import sys

# Test maturin command line interface
print('Testing maturin CLI interface...')

# Check maturin version via CLI
result = subprocess.run([sys.executable, '-m', 'maturin', '--version'], 
                      capture_output=True, text=True, timeout=10)

assert result.returncode == 0, f'maturin --version exited {result.returncode}: {result.stderr.strip()}'
assert result.stdout.startswith('maturin '), result.stdout
print(f'maturin CLI version: {result.stdout.strip()}')
import importlib.metadata
import maturin  # the PEP 517 backend module shipped with the CLI, imported directly
assert callable(maturin.build_wheel) and callable(maturin.build_sdist)
assert result.stdout.strip() == f"maturin {importlib.metadata.version('maturin')}", result.stdout

# Check help information
help_result = subprocess.run([sys.executable, '-m', 'maturin', '--help'], 
                            capture_output=True, text=True, timeout=10)

assert help_result.returncode == 0, f'maturin --help exited {help_result.returncode}: {help_result.stderr.strip()}'
help_lines = help_result.stdout.split('\n')[:10]
print('maturin help (first 10 lines):')
for line in help_lines:
    print(f'  {line}')

print('Example 1: CLI interface test completed successfully')


## Example 2: Project Structure Analysis

Analyze and demonstrate maturin project structure requirements.

In [ ]:
# Example 2: Project structure analysis
import tempfile
import os

# Create temporary directory to demonstrate project structure
with tempfile.TemporaryDirectory() as temp_dir:
    print(f'Creating sample maturin project structure in: {temp_dir}')
    
    # Create basic maturin project files
    project_files = {
        'Cargo.toml': '''[package]
name = "example-rust-ext"
version = "0.1.0"
edition = "2021"

[lib]
crate-type = ["cdylib"]

[dependencies]
pyo3 = { version = "0.20", features = ["extension-module"] }
''',
        'pyproject.toml': '''[build-system]
requires = ["maturin>=1.0,<2.0"]
build-backend = "maturin"

[project]
name = "example-rust-ext"
version = "0.1.0"
description = "Example Rust extension"
''',
        'src/lib.rs': '''use pyo3::prelude::*;

#[pyfunction]
fn add_numbers(a: i64, b: i64) -> i64 {
    a + b
}

#[pymodule]
fn example_rust_ext(_py: Python, m: &PyModule) -> PyResult<()> {
    m.add_function(wrap_pyfunction!(add_numbers, m)?)?;
    Ok(())
}
'''
    }
    
    # Write sample files
    for filename, content in project_files.items():
        filepath = os.path.join(temp_dir, filename)
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        
        with open(filepath, 'w') as f:
            f.write(content)
        print(f'Created: {filename}')
    
    # List project structure
    print('\nProject structure created:')
    for root, dirs, files in os.walk(temp_dir):
        level = root.replace(temp_dir, '').count(os.sep)
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/')
        sub_indent = ' ' * 2 * (level + 1)
        for file in files:
            print(f'{sub_indent}{file}')
    
    print('Example 2: Project structure analysis completed successfully')

## Example 3: Build Configuration and Commands

Demonstrate maturin build configuration and command options.

In [ ]:
# Example 3: Build configuration and commands
import subprocess
import sys

print('Testing maturin build configurations...')

# Test different maturin commands and options
commands_to_test = [
    ['--help'],
    ['list-python'],  # List available Python interpreters
    ['new', '--help'],  # Help for creating new projects
    ['build', '--help'],  # Help for build command
    ['develop', '--help']  # Help for develop command
]

for cmd in commands_to_test:
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'maturin'] + cmd,
            capture_output=True, text=True, timeout=5
        )
        
        if result.returncode == 0:
            output_lines = result.stdout.split('\n')[:5]  # First 5 lines
            print(f'\nmaturin {" ".join(cmd)}:')
            for line in output_lines:
                if line.strip():
                    print(f'  {line}')
        else:
            print(f'\nmaturin {" ".join(cmd)}: {result.stderr.strip()[:100]}')
            
    except subprocess.TimeoutExpired:
        print(f'\nmaturin {" ".join(cmd)}: Command timed out')
    except Exception as e:
        print(f'\nmaturin {" ".join(cmd)}: Error - {e}')
        raise

# Test configuration parsing
config_examples = {
    'build_backend': 'maturin',
    'requires': ['maturin>=1.0'],
    'build_options': ['--release', '--strip']
}

print(f'\nExample build configuration:')
for key, value in config_examples.items():
    print(f'  {key}: {value}')

print('Example 3: Build configuration completed successfully')

## Example 4: PyO3 Integration Patterns

Demonstrate PyO3 integration patterns commonly used with maturin.

In [ ]:
# Example 4: PyO3 integration patterns
print('Demonstrating PyO3 integration patterns for maturin projects...')

# Common PyO3 function patterns for Rust extensions
pyo3_patterns = {
    'simple_function': '''#[pyfunction]
fn multiply(a: f64, b: f64) -> f64 {
    a * b
}''',
    
    'class_with_methods': '''#[pyclass]
struct Calculator {
    value: f64,
}

#[pymethods]
impl Calculator {
    #[new]
    fn new(initial: f64) -> Self {
        Calculator { value: initial }
    }
    
    fn add(&mut self, other: f64) {
        self.value += other;
    }
}''',
    
    'error_handling': '''#[pyfunction]
fn safe_divide(a: f64, b: f64) -> PyResult<f64> {
    if b == 0.0 {
        Err(PyErr::new::<pyo3::exceptions::ZeroDivisionError, _>(
            "Cannot divide by zero"
        ))
    } else {
        Ok(a / b)
    }
}''',
    
    'module_definition': '''#[pymodule]
fn my_extension(_py: Python, m: &PyModule) -> PyResult<()> {
    m.add_function(wrap_pyfunction!(multiply, m)?)?;
    m.add_class::<Calculator>()?;
    m.add_function(wrap_pyfunction!(safe_divide, m)?)?;
    Ok(())
}'''
}

# Display pattern examples
for pattern_name, code in pyo3_patterns.items():
    print(f'\n{pattern_name.replace("_", " ").title()}:')
    for line in code.split('\n'):
        print(f'  {line}')

# Configuration for different binding types
binding_types = {
    'pyo3': 'Pure PyO3 bindings',
    'pyo3-mixed': 'PyO3 with Python code',
    'cffi': 'CFFI bindings (legacy)',
    'uniffi': 'UniFFI bindings'
}

print(f'\nSupported binding types:')
for binding, description in binding_types.items():
    print(f'  {binding}: {description}')

print('Example 4: PyO3 integration patterns completed successfully')

## Example 5: Publishing and Deployment Workflow

Demonstrate maturin publishing and deployment workflow patterns.

In [ ]:
# Example 5: Publishing and deployment workflow
import subprocess
import sys

print('Demonstrating maturin publishing and deployment workflow...')

# Test publishing-related commands
publishing_commands = [
    ['publish', '--help'],
    ['upload', '--help'],
    ['build', '--release', '--help']
]

for cmd in publishing_commands:
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'maturin'] + cmd,
            capture_output=True, text=True, timeout=3
        )
        
        if result.returncode == 0:
            # Extract key options from help
            lines = result.stdout.split('\n')
            key_options = [line for line in lines if line.strip().startswith('-') and len(line) < 80][:3]
            print(f'\nmaturin {cmd[0]} key options:')
            for option in key_options:
                print(f'  {option.strip()}')
        else:
            print(f'\nmaturin {cmd[0]}: Command not fully available')
            
    except Exception as e:
        print(f'\nmaturin {cmd[0]}: {str(e)[:50]}')
        raise

# Deployment workflow steps
workflow_steps = {
    1: 'maturin build --release  # Build optimized wheel',
    2: 'maturin build --sdist    # Build source distribution',  
    3: 'maturin publish         # Publish to PyPI',
    4: 'maturin upload          # Upload pre-built wheels'
}

print(f'\nTypical deployment workflow:')
for step, command in workflow_steps.items():
    print(f'  {step}. {command}')

# CI/CD integration examples
ci_examples = {
    'GitHub Actions': 'uses: PyO3/maturin-action@v1',
    'Build matrix': 'python-version: [3.8, 3.9, 3.10, 3.11]',
    'Cross compilation': '--target x86_64-unknown-linux-gnu'
}

print(f'\nCI/CD integration patterns:')
for platform, example in ci_examples.items():
    print(f'  {platform}: {example}')

print('Example 5: Publishing and deployment workflow completed successfully')